# Constraint Detection

In [22]:
# Import neccessay libraries
import pandas as pd
import numpy as np

In [23]:
# Load the dataset
from pathlib import Path

# Path.cwd() gets the 'modeling' directory, and .parent moves up to the project root
project_root = Path.cwd().parent

# The division operator (/) is used by pathlib to join paths safely on any OS
outlets_path = project_root / "lakehouse" / "silver" / "outlet_master.parquet"
transactions_path = project_root / "lakehouse" / "silver" / "transactions_history.parquet"

# Read the parquet file into a DataFrame
outlets = pd.read_parquet(outlets_path)
tx = pd.read_parquet(transactions_path)

In [24]:
# Coefficient of Variation (CV) for each outlet

# Undertanding th e celings of outlets by looking at the coefficient of variation (CV) of their sales volumes.

outlet_stats = tx.groupby('Outlet_ID')['Volume_Liters'].agg(
    mean_vol   = 'mean',
    max_vol    = 'max',
    min_vol    = 'min',
    std_vol    = 'std',
    n_months   = 'count',
    p90_vol    = lambda x: x.quantile(0.90),
    p10_vol    = lambda x: x.quantile(0.10),
    ).reset_index()
outlet_stats['CV'] = outlet_stats['std_vol'] / outlet_stats['mean_vol']
outlet_stats['utilization'] = outlet_stats['mean_vol'] / (outlet_stats['max_vol'] + 1e-8)  # Avoid division by zero



In [25]:
# understanding ceiling hitting rate

def ceiling_hitting_rate(series):
    max_val = series.max()
    near_celling = (series >= max_val * 0.95).sum()  # Count how many times sales are within 95% of max
    return near_celling / len(series)

ceiling_rate = tx.groupby('Outlet_ID')['Volume_Liters'].apply(ceiling_hitting_rate).reset_index()
ceiling_rate.columns = ['Outlet_ID', 'Ceiling_Hitting_Rate']

outlet_stats = outlet_stats.merge(ceiling_rate, on='Outlet_ID')

# outlets with high ceiling hitting rates more than 3 times

suspicious_outlets = ceiling_rate[(ceiling_rate['Ceiling_Hitting_Rate'] > 0.3) ]
print(f"Number of suspicious outlets: {len(suspicious_outlets)}")

Number of suspicious outlets: 0


In [26]:
# Discovering zero month rate

zero_rate = tx.groupby('Outlet_ID')['Volume_Liters'].apply(
    lambda x: (x == 0).sum() / len(x)
).reset_index()
zero_rate.columns = ['Outlet_ID', 'Zero_Month_Rate']

outlet_stats = outlet_stats.merge(zero_rate, on='Outlet_ID')

In [30]:
# Composite constraint score (simple average of CV, ceiling rate, and zero month rate)
outlet_stats['Constraint_Score'] = (
    (outlet_stats['CV'] < 0.15).astype(int) * 0.3 + # Low CV suggests consistent sales, which can indicate a credit cap
    (outlet_stats['Ceiling_Hitting_Rate'] > 0.4).astype(int) * 0.4 + # High ceiling hitting rate suggests hitting a sales ceiling repeatedly
    (outlet_stats['Zero_Month_Rate'] < 0.3).astype(int) * 0.25  # Low zero month rate suggests consistent sales, which can indicate a credit cap
) 

outlet_stats['is_constrained'] = outlet_stats['Constraint_Score'] > 0.4 # Threshold can be tuned based on how many outlets we want to flag

print(f"Constrained outlets: {outlet_stats['is_constrained'].sum()} out of {len(outlet_stats)}")

Constrained outlets: 0 out of 19960


In [32]:
outlet_stats.to_parquet("../lakehouse/silver/outlet_stats.parquet")
print("Outlet stats saved to silver layer.")

Outlet stats saved to silver layer.
